# PyEIDORS/DOLFINx：经典 CEM 与 Robin CEM / Classic CEM versus Robin CEM

## 目标 / Goal

| 中文 | English |
|---|---|
| 从头到尾运行一个有理数精确案例，检查 DOLFINx 组装的 $A_R,C,D$ 块，分别求解传统增广 CEM 与约化 Robin/跨导 CEM，再与经过认证的有理数域 $\mathbb{Q}$ 精确电极电压比较，最后复现 38 个案例的报告汇总。 | Run one exact-rational case from top to bottom, inspect the DOLFINx-assembled $A_R,C,D$ blocks, solve the traditional augmented CEM and the reduced Robin/transconductance CEM, compare both with the certified exact voltage over $\mathbb{Q}$, and reproduce the 38-case report summary. |

| 默认案例 | Default case |
|---|---|
| `X01` 足够小，适合交互式调试和逐个查看矩阵。 | `X01` is intentionally small enough for interactive debugging and matrix inspection. |


## 设置 / Setup

| 中文 | English |
|---|---|
| 必须从仓库根目录进入真实数 `float64` Nix 环境。 | Start from the repository root in the real-valued `float64` Nix profile. |

```bash
nix develop .#default --command jupyter lab \
  examples/cem_exact_extension_walkthrough/pyeidors_walkthrough.ipynb
```

| 中文 | English |
|---|---|
| VS Code 中先运行 `nix develop .#default --command python examples/cem_exact_extension_walkthrough/register_vscode_kernel.py`，重载窗口后选择 **Jupyter Kernel → PyEIDORS real float64 (Nix)**。不要选择裸的 `/nix/store/.../bin/python`。 | In VS Code, first run `nix develop .#default --command python examples/cem_exact_extension_walkthrough/register_vscode_kernel.py`, reload the window, and select **Jupyter Kernel → PyEIDORS real float64 (Nix)**. Do not select the raw `/nix/store/.../bin/python`. |

| 参数 | 中文 | English |
|---|---|---|
| `REGENERATE=False` | 复用已有认证结果，适合首次讲解。 | Reuse the certified result; recommended for the first walkthrough. |
| `REGENERATE=True` | 强制重新执行 DOLFINx 组装。 | Force a fresh DOLFINx assembly. |


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "cem_exact_extension_walkthrough":
    NOTEBOOK_DIR = Path("examples/cem_exact_extension_walkthrough").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
for path in (REPO_ROOT, REPO_ROOT / "src", NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from matplotlib import font_manager  # noqa: E402

from experiment_common import (  # noqa: E402
    build_classic_state,
    build_robin_state,
    exact_reference_metrics,
    formulation_diagnostics,
    load_assembled_blocks,
    load_csv_records,
    load_forward_fixture,
    load_portable_exact_reference,
    plot_forward_fixture,
    plot_forward_solution,
    solve_classic,
    solve_robin,
    summarize_accuracy_records,
)
from pyeidors_debug import ensure_pyeidors_case  # noqa: E402

for font_path in (
    Path("/mnt/c/Windows/Fonts/times.ttf"),
    Path("/mnt/c/Windows/Fonts/timesbd.ttf"),
    Path("/mnt/c/Windows/Fonts/msyh.ttc"),
):
    if font_path.exists():
        font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["Times New Roman", "Microsoft YaHei"]

In [2]:
# 选择案例和是否重新组装 / Select the case and whether to reassemble.
CASE_ID = "X01"
REGENERATE = False
SUITE_OUTPUT = REPO_ROOT / "output" / "cem_exact_extension"
REFERENCE_PATH = NOTEBOOK_DIR / "fixtures" / "X01" / "exact_reference.json"
METRICS_PATH = NOTEBOOK_DIR / "expected" / "cem_exact_extension_metrics.csv"
FIGURE_DIR = NOTEBOOK_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 符号与变量字典 / Symbol and variable dictionary

| 变量或符号 | 中文含义 | English meaning | 类型或维度 |
|---|---|---|---|
| `CASE_ID` | 当前正问题案例编号；默认 `X01` 是均匀背景电导率案例。 | Selected forward-problem case; default `X01` has uniform background conductivity. | `str` |
| `fixture` / `metadata_path` | 三个框架共用的规范输入：节点、三角形、电极边、逐单元电导率、接触阻抗和电流模式。 | Canonical input shared by all three frameworks: nodes, triangles, electrode edges, cell conductivity, contact impedance, and drives. | 路径和元数据 / paths and metadata |
| `PyEIDORS/DOLFINx_report` | PyEIDORS/DOLFINx 正式运行器保存的求解器、离散化、计时和原始电压报告。 | Solver, discretization, timing, and raw-voltage report saved by the official PyEIDORS/DOLFINx runner. | `dict` |
| $N$ | 有限元体节点数。 | Number of FEM body nodes. | `int` |
| $K$ | 三角形单元数。 | Number of triangular cells. | `int` |
| $L$ | 电极数。 | Number of electrodes. | `int` |
| $P$ | 同时求解的电流模式（右端项）数。 | Number of current patterns/right-hand sides. | `int` |
| `nodes` | 每个体节点的 $(x,y)$ 坐标。 | $(x,y)$ coordinate of every body node. | $N\times2$ |
| `cells` | 每个 P1 三角形的三个节点索引。 | Three node indices of every P1 triangle. | $K\times3$ |
| `tagged_edges` | 边界边的两个节点及其电极标签；标签 0 表示绝缘边。 | Two vertices and electrode label for each boundary edge; label 0 is insulating. | $E_b\times3$ |
| `cell_conductivity`, $\sigma_k$ | 单元电导率：第 $k$ 个三角形内的常数值；X01 全部为 $1/8$。 | Cell conductivity: the constant value in cell $k$; every X01 cell is $1/8$. | $K$ 个标量 / $K$ scalars |
| `contact_impedance`, $z_\ell$ | 第 $\ell$ 个电极的接触阻抗；X01 为 1。 | Contact impedance of electrode $\ell$; X01 uses 1. | $L$ 个标量 / $L$ scalars |
| `blocks.currents`, $I$ | 边界注入电流；每一列严格满足 $\mathbf{1}^\mathsf{T}I_{:,p}=0$。 | Injected current on the boundary; every column satisfies $\mathbf{1}^\mathsf{T}I_{:,p}=0$. | $L\times P$ |
| `blocks.robin_matrix`, $A_R$ | 体刚度矩阵加电极 Robin 边界质量项。 | Body stiffness matrix plus the electrode Robin boundary mass. | N×N 稀疏矩阵 |
| `blocks.coupling`, $C$ | 体节点自由度与常数电极电势之间的耦合。 | Coupling between body nodal DOFs and constant electrode voltages. | N×L 稀疏矩阵 |
| `blocks.electrode_matrix`, $D$ | 电极边界积分形成的电极块。 | Electrode block from boundary integrals. | L×L 稀疏矩阵 |
| $u$ | 每个电流模式对应的体内节点电势。 | Body nodal potential for each drive. | $N$×$P$ |
| $U$ | 每个电流模式对应的边界电极电压，也是本实验比较的主要输出。 | Boundary electrode voltage for each drive; the principal compared output. | $L$×$P$ |
| $\lambda$ | 传统 CEM 中实施 $\mathbf{1}^\mathsf{T}U=0$ 的拉格朗日乘子。 | Gauge multiplier enforcing $\mathbf{1}^\mathsf{T}U=0$ in Classic CEM. | $1$×$P$ |
| $Q$ | `float64` 求解中电极零和子空间的正交基。 | Orthonormal basis of the zero-sum electrode subspace in the `float64` solve. | $L$×$(L-1)$ |
| $y$ | 电极电压在零和基中的坐标，满足 $U=Qy$。 | Coordinates of electrode voltage in the zero-sum basis, with $U=Qy$. | $(L-1)$×$P$ |
| `response_basis`, $R$ | 解 $A_RR=CQ$ 得到的体响应基；代码用分解求解，不显式形成逆矩阵。 | Body response basis from $A_RR=CQ$; solved by a factorization without forming an inverse. | $N$×$(L-1)$ |
| `schur_action_basis` | $DQ-C^\mathsf{T}R$，即完整跨导算子作用在 $Q$ 上的结果。 | $DQ-C^\mathsf{T}R$, the full transconductance action on $Q$. | $L$×$(L-1)$ |
| `reduced_map`, $T_r$ | $Q^\mathsf{T}(DQ-C^\mathsf{T}R)$，Robin CEM 实际分解的小矩阵。 | $Q^\mathsf{T}(DQ-C^\mathsf{T}R)$, the small matrix actually factored by Robin CEM. | $(L-1)$×$(L-1)$ |
| `nnz` | 稀疏矩阵中非零元素数量，不是误差。 | Number of stored nonzeros in a sparse matrix; it is not an error metric. | `int` |
| `mesh_fingerprint` | 网格指纹：对规范节点、单元和带标签边界边计算的 SHA-256；三个报告必须一致。 | Mesh fingerprint: SHA-256 of canonical nodes, cells, and tagged boundary edges; all three reports must match it. | 64 位十六进制 / hex chars |

### 38 个案例究竟计算什么 / What the 38 cases compute

每个案例都是 **CEM 正问题**，不是逆问题重构：给定同一案例的网格、
逐单元电导率、电极、接触阻抗和零和电流模式，分别用传统 CEM 与 Robin CEM
计算体内节点电势 $u$ 和边界电极电压 $U$。每个框架必须先画出自己实际加载的
网格、$\sigma_k$ 和选定的边界注流，并通过相同网格指纹认证，之后结果才进入
跨框架比较。

Every case is a **CEM forward problem**, not an inverse reconstruction:
given the case's mesh, cell conductivity, electrodes, contact impedance, and
zero-sum drives, Classic and Robin CEM compute body potential $u$ and boundary
electrode voltage $U$. Each framework must first display the mesh,
$\sigma_k$, and a selected boundary drive that it actually loaded, and must
certify the same mesh fingerprint before its result enters the cross-framework
comparison.

| 案例 | 正问题设置 / Forward setting | 主要用途 / Purpose |
|---|---|---|
| X01–X16 | Q0/Q2 网格；16 电极；均匀有理 $\sigma$；多种有理 $z$；相邻和 skip-4 注流。 | 改变网格、物性范围、接触阻抗和注流跨度。 / Vary mesh, physical range, impedance, and drive span. |
| X17–X24 | Q0/Q2 网格；16 电极；左右两区 $\sigma=1/4$ 与 $1$；两种 $z$ 和注流。 | 表示已知内部非均匀待测物后的正向边界电压。 / Forward voltages for a known internal heterogeneity. |
| X25–X32 | Q0/Q2 网格；8 电极；均匀 $\sigma=1/4$；两种 $z$ 和注流。 | 检查电极数改变。 / Check a different electrode count. |
| X33–X38 | 更细 Q4 网格；16 电极；均匀 $\sigma=1/4$；三种 $z$ 和两种注流。 | 检查更大有理离散系统。 / Check the larger rational discretization. |


## 步骤 / Steps

### 1. 获取共享案例与 PyEIDORS 分块 / Obtain the shared case and PyEIDORS blocks

| 中文 | English |
|---|---|
| 共享夹具固定了节点、三角形、电极边、逐单元电导率、接触阻抗、电流模式、P1 阶次和真实数 `float64`。三个 FEM 框架导入完全相同的规范数据。 | The shared fixture fixes nodes, triangles, electrode edges, per-cell conductivity, contact impedance, current patterns, P1 order, and real `float64`. All three FEM frameworks import the same canonical data. |


In [3]:
fixture, pyeidors_report = ensure_pyeidors_case(
    CASE_ID,
    SUITE_OUTPUT,
    regenerate=REGENERATE,
)
block_path = Path(fixture["case_dir"]) / "pyeidors_assembled_blocks.mat"
blocks = load_assembled_blocks(block_path)
forward_fixture = load_forward_fixture(
    Path(fixture["mat_path"]),
    Path(fixture["metadata_path"]),
)
solver_mesh_fingerprint = pyeidors_report["discretization"]["mesh_fingerprint"]
assert forward_fixture.mesh_fingerprint == solver_mesh_fingerprint

{
    "case": CASE_ID,
    "N_nodes": forward_fixture.nodes.shape[0],
    "K_cells": forward_fixture.cells.shape[0],
    "L_electrodes": forward_fixture.electrode_count,
    "P_current_patterns": forward_fixture.currents.shape[1],
    "potential_order": forward_fixture.potential_order,
    "scalar_dtype": forward_fixture.scalar_dtype,
    "conductivity_pattern": forward_fixture.conductivity_pattern,
    "unique_cell_conductivity": np.unique(forward_fixture.cell_conductivity),
    "contact_impedance_exact": forward_fixture.contact_impedance_exact,
    "first_current_pattern": forward_fixture.currents[:, 0],
    "current_column_sums": np.sum(forward_fixture.currents, axis=0),
    "A_R_shape": blocks.robin_matrix.shape,
    "A_R_nnz": blocks.robin_matrix.nnz,
    "C_shape": blocks.coupling.shape,
    "D_shape": blocks.electrode_matrix.shape,
    "I_shape": blocks.currents.shape,
    "mesh_fingerprint": solver_mesh_fingerprint,
}

{'case': 'X01',
 'N_nodes': 33,
 'K_cells': 32,
 'L_electrodes': 16,
 'P_current_patterns': 16,
 'potential_order': 1,
 'scalar_dtype': 'float64',
 'conductivity_pattern': 'uniform',
 'unique_cell_conductivity': array([0.125]),
 'contact_impedance_exact': '1',
 'first_current_pattern': array([ 1., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.]),
 'current_column_sums': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 'A_R_shape': (33, 33),
 'A_R_nnz': 161,
 'C_shape': (33, 16),
 'D_shape': (16, 16),
 'I_shape': (16, 16),
 'mesh_fingerprint': '7be7165ad3bdd3661ae06bea768622741ece609acde5720f3dd0c0cbde85c5bc'}

### 2. 显示公平的正问题条件 / Display the fair forward conditions

| 中文 | English |
|---|---|
| 左图显示实际加载的 P1 网格和每个三角形的电导率；右图在完全相同的边界上显示第一个电流模式的注入电极 `+I` 与回流电极 `−I`。X01 是均匀背景正问题，所以没有内部异常物；求解目标是在给定 $\sigma,z,I$ 后计算体电势 $u$ 与边界电压 $U$。 | The left panel shows the loaded P1 mesh and conductivity in every triangle. The right panel shows the injecting `+I` and returning `−I` electrodes for the first drive on the same boundary. X01 is a uniform-background forward problem with no interior anomaly; its task is to compute body potential $u$ and boundary voltage $U$ for prescribed $\sigma,z,I$. |
| 图题中的网格指纹来自规范节点、三角形和电极边；上一个单元已经断言它与 PyEIDORS 报告一致。 | The mesh fingerprint in the title is computed from canonical nodes, triangles, and electrode edges; the previous cell asserted that it matches the PyEIDORS report. |


In [4]:
fairness_figure, fairness_axes = plot_forward_fixture(
    forward_fixture,
    current_column=0,
)
fairness_figure.savefig(
    FIGURE_DIR / f"{CASE_ID}_pyeidors_forward_setup.png",
    dpi=180,
    bbox_inches="tight",
)
fairness_figure

<Figure size 1320x580 with 3 Axes>

### 3. 传统经典 CEM / Traditional Classic CEM

| 中文 | English |
|---|---|
| 经典方法把体内节点电势 $u$、电极电势 $U$ 和零均值约束的拉格朗日乘子 $\lambda$ 放入一个增广线性系统，一次分解后求解全部电流右端项。 | The Classic method places body potentials $u$, electrode voltages $U$, and the zero-mean gauge multiplier $\lambda$ in one augmented linear system, then solves all current right-hand sides after one factorization. |

$$
\begin{bmatrix}
A_R & C & 0 \\
C^\mathsf{T} & D & \mathbf{1} \\
0 & \mathbf{1}^\mathsf{T} & 0
\end{bmatrix}
\begin{bmatrix}
u \\ U \\ \lambda
\end{bmatrix}
=
\begin{bmatrix}
0 \\ I \\ 0
\end{bmatrix}.
$$

| 中文 | English |
|---|---|
| 在下一单元后暂停，重点查看 `classic_state.system_matrix`、`classic_state.factor`、`classic_solution.body_potential` 和 `classic_solution.electrode_voltage`。 | Stop after the next cell and inspect `classic_state.system_matrix`, `classic_state.factor`, `classic_solution.body_potential`, and `classic_solution.electrode_voltage`. |


In [5]:
classic_state = build_classic_state(blocks)
classic_solution = solve_classic(classic_state, blocks.currents)

{
    "augmented_shape": classic_state.system_matrix.shape,
    "augmented_nnz": classic_state.system_matrix.nnz,
    "body_potential_shape": classic_solution.body_potential.shape,
    "electrode_voltage_shape": classic_solution.electrode_voltage.shape,
}

{'augmented_shape': (50, 50),
 'augmented_nnz': 273,
 'body_potential_shape': (33, 16),
 'electrode_voltage_shape': (16, 16)}

### 4. Robin/跨导 CEM / Robin/transconductance CEM

| 中文 | English |
|---|---|
| $Q$ 是电极零和子空间的正交基，即 $Q^\mathsf{T}Q=I$ 且 $Q^\mathsf{T}\mathbf{1}=0$。先分解 $A_R$，消去体内未知量，再只在 $L-1$ 维电极子空间求解。 | $Q$ is an orthonormal basis of the zero-sum electrode subspace: $Q^\mathsf{T}Q=I$ and $Q^\mathsf{T}\mathbf{1}=0$. Factor $A_R$, eliminate the body unknowns, and solve only on the $L-1$ dimensional electrode subspace. |

$$
R=A_R^{-1}CQ,\qquad
T_r=Q^\mathsf{T}\left(DQ-C^\mathsf{T}R\right)
=Q^\mathsf{T}\left(D-C^\mathsf{T}A_R^{-1}C\right)Q.
$$

$$
T_r y=Q^\mathsf{T}I,\qquad
U=Qy,\qquad
u=-Ry.
$$

| 中文 | English |
|---|---|
| 代码不会直接求逆完整的奇异跨导矩阵；`response_basis` 对应 $R$，`schur_action_basis` 对应 $DQ-C^\mathsf{T}R$，`reduced_map` 对应 $T_r$。 | The code never directly inverts the full singular transconductance matrix; `response_basis` is $R$, `schur_action_basis` is $DQ-C^\mathsf{T}R$, and `reduced_map` is $T_r$. |


In [6]:
robin_state = build_robin_state(blocks)
robin_solution = solve_robin(robin_state, blocks.currents)

{
    "Q_shape": robin_state.electrode_basis.shape,
    "response_basis_shape": robin_state.response_basis.shape,
    "Schur_action_shape": robin_state.schur_action_basis.shape,
    "reduced_map_shape": robin_state.reduced_map.shape,
    "Q_orthogonality_error": np.linalg.norm(
        robin_state.electrode_basis.T @ robin_state.electrode_basis
        - np.eye(blocks.electrode_count - 1)
    ),
    "Q_zero_sum_error": np.linalg.norm(
        np.ones(blocks.electrode_count) @ robin_state.electrode_basis
    ),
    "reduced_condition_number": np.linalg.cond(robin_state.reduced_map),
}

{'Q_shape': (16, 15),
 'response_basis_shape': (33, 15),
 'Schur_action_shape': (16, 15),
 'reduced_map_shape': (15, 15),
 'Q_orthogonality_error': np.float64(5.1359056776231935e-16),
 'Q_zero_sum_error': np.float64(4.2998752849492583e-16),
 'reduced_condition_number': np.float64(3.9724054545232192)}

### 5. 比较两条浮点计算路径 / Compare the two floating-point routes

| 中文 | English |
|---|---|
| 两种方法在精确算术下等价，但矩阵分解、消元和乘法顺序不同，因此 `float64` 结果可能存在舍入级差异。 | The two methods are equivalent in exact arithmetic, but different factorization, elimination, and multiplication orders can produce roundoff-level differences in `float64`. |


In [7]:
solutions = {
    "classic": classic_solution,
    "robin_transconductance": robin_solution,
}
diagnostics = formulation_diagnostics(blocks, solutions)
diagnostics

{'electrode_voltage_relative_l2': 7.120845370859045e-16,
 'body_potential_relative_l2': 1.882311028705559e-15,
 'classic_scaled_backward_residual': 1.346141744708027e-17,
 'robin_scaled_backward_residual': 4.709065260146326e-17,
 'classic_voltage_gauge_max_abs': 1.7763568394002505e-15,
 'robin_voltage_gauge_max_abs': 4.440892098500626e-15}

### 6. 求解结果可视化 / Forward-result visualization

| 中文 | English |
|---|---|
| 上排使用同一色标显示第一个注流模式的 Classic 体电势、Robin 体电势和体电势差值 `Robin − Classic`；下排使用同一电压纵轴显示两条电极电压曲线，并单独放大舍入级电压差。 | The top row uses shared limits for the Classic body potential, Robin body potential, and signed `Robin − Classic` body-potential difference of the first drive. The bottom row uses shared voltage limits for both electrode traces and separately magnifies their roundoff-level voltage difference. |
| 这些图直接读取前面用于诊断的 `classic_solution` 与 `robin_solution`，没有重新求解或替换数据。 | These figures read the same `classic_solution` and `robin_solution` arrays used by the diagnostics; they do not rerun or replace the solve. |


In [8]:
result_figure, result_axes = plot_forward_solution(
    forward_fixture,
    solutions,
    current_column=0,
)
result_figure.savefig(
    FIGURE_DIR / f"{CASE_ID}_pyeidors_classic_robin_results.png",
    dpi=180,
    bbox_inches="tight",
)
result_figure

<Figure size 1580x940 with 8 Axes>

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].spy(classic_state.system_matrix, markersize=1.1, color="#475569")
axes[0].set_title("经典 CEM 增广矩阵 / Classic augmented CEM matrix")
axes[1].imshow(robin_state.reduced_map, cmap="cividis", aspect="auto")
axes[1].set_title("Robin 约化映射 / Robin reduced map $T_r$")
fig.tight_layout()

Font 'default' does not have a glyph for '\u7ea6' [U+7ea6], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u5316' [U+5316], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u6620' [U+6620], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u5c04' [U+5c04], substituting with a dummy symbol.


### 7. 与有理数精确电压比较 / Compare with the exact rational voltage

| 中文 | English |
|---|---|
| X01 参考文件保存分数而不是舍入后的小数。候选 `float64` 数值先被解释为其精确二进制有理数，再以 100 位精度计算误差，因此评估过程不会再引入普通双精度减法误差。 | The X01 reference stores fractions rather than rounded decimal truth. Candidate `float64` values are promoted to their exact binary rational values before the 100-digit metric evaluation, so the metric calculation does not introduce another ordinary double-precision subtraction error. |

$$
\varepsilon_{\mathrm{truth}}
=\frac{\left\|U_{\mathrm{float64}}-U_{\mathbb{Q}}\right\|_F}
{\left\|U_{\mathbb{Q}}\right\|_F}.
$$


In [10]:
exact_reference = load_portable_exact_reference(REFERENCE_PATH)
exact_metrics = {
    name: exact_reference_metrics(solution.electrode_voltage, exact_reference)
    for name, solution in solutions.items()
}
exact_metrics

{'classic': {'truth_relative_l2': 1.3032086527364565e-15,
  'truth_max_abs': 9.345317203799345e-15,
  'exact_reduced_scaled_backward_residual': 4.5628231485871555e-17,
  'voltage_gauge_relative_residual': 6.823220871811818e-17,
  'reduced_condition_number_2_estimate': 47.697807812864},
 'robin_transconductance': {'truth_relative_l2': 1.4291676735987542e-15,
  'truth_max_abs': 1.0011451018574439e-14,
  'exact_reduced_scaled_backward_residual': 4.1461400398838096e-17,
  'voltage_gauge_relative_residual': 1.3722035304554455e-16,
  'reduced_condition_number_2_estimate': 47.697807812864}}

## 检查 / Checks

| 中文 | English |
|---|---|
| 只有经典有理数系统残差与 Robin 有理数系统残差都严格等于零、两种精确电压完全相同且电压规范残差严格为零时，参考解才被认证。 | The exact reference is certified only when the Classic rational residual and Robin rational residual are both exactly zero, both exact voltages are identical, and the voltage gauge residual is exactly zero. |


In [11]:
exact_reference["certification"]

{'exact_classic_residual_zero': True,
 'exact_robin_residual_zero': True,
 'exact_classic_robin_identical': True,
 'exact_voltage_gauge_zero': True}

In [12]:
stored_voltages = {
    name: np.asarray(values, dtype=np.float64)
    for name, values in pyeidors_report["raw_electrode_voltages"].items()
}
assert np.array_equal(
    classic_solution.electrode_voltage,
    stored_voltages["classic"],
)
assert np.array_equal(
    robin_solution.electrode_voltage,
    stored_voltages["robin_transconductance"],
)
assert exact_reference["certification"]["exact_classic_residual_zero"]
assert exact_reference["certification"]["exact_robin_residual_zero"]
assert exact_reference["certification"]["exact_classic_robin_identical"]
"All selected-case checks passed."

'All selected-case checks passed.'

### 复现 38 个案例的报告数字 / Reproduce the 38-case report numbers

| 中文 | English |
|---|---|
| 下面的单元从冻结的 228 条精度记录重新计算几何平均误差、逐案例胜出次数和 Q4 网格排序。 | The next cell recomputes geometric-mean errors, per-case win counts, and the Q4 ordering from the 228 frozen accuracy records. |


In [13]:
summary = summarize_accuracy_records(load_csv_records(METRICS_PATH))
{
    "record_count": summary["record_count"],
    "case_count": summary["case_count"],
    "geometric_means": summary["geometric_means"],
    "win_counts": summary["win_counts"],
    "q4_summary": summary["q4_summary"],
}

{'record_count': 228,
 'case_count': 38,
 'geometric_means': {'classic': {'EIDORS': 1.6935476890754599e-15,
   'NGSolve': 1.1200923250127873e-14,
   'PyEIDORS/DOLFINx': 1.1094109141135253e-15},
  'robin_transconductance': {'EIDORS': 1.7352953602888227e-15,
   'NGSolve': 1.0751276317956813e-14,
   'PyEIDORS/DOLFINx': 1.0595913702941103e-15}},
 'win_counts': {'classic': {'EIDORS': 11,
   'NGSolve': 0,
   'PyEIDORS/DOLFINx': 27},
  'robin_transconductance': {'EIDORS': 9,
   'NGSolve': 0,
   'PyEIDORS/DOLFINx': 29}},
 'q4_summary': {'classic': {'case_ids': ['X33',
    'X34',
    'X35',
    'X36',
    'X37',
    'X38'],
   'same_order_all_cases': True,
   'ordering': ['PyEIDORS/DOLFINx', 'EIDORS', 'NGSolve']},
  'robin_transconductance': {'case_ids': ['X33',
    'X34',
    'X35',
    'X36',
    'X37',
    'X38'],
   'same_order_all_cases': True,
   'ordering': ['PyEIDORS/DOLFINx', 'EIDORS', 'NGSolve']}}}

## 后续步骤 / Next Steps

| 中文 | English |
|---|---|
| 1. 完成完整 `prepare` 后可以修改 `CASE_ID`。<br>2. 设置 `REGENERATE=True` 检查 DOLFINx 组装。<br>3. 在 `pyeidors_debug.py` 中设置断点逐行查看变量。<br>4. 对同一 MSH/JSON 运行 NGSolve Notebook，并对同一 MAT 运行 MATLAB 脚本。<br>5. 按 `README.md` 运行完整 `compare` 与 `timing`。 | 1. Change `CASE_ID` after the full `prepare` step.<br>2. Set `REGENERATE=True` to inspect DOLFINx assembly.<br>3. Set breakpoints in `pyeidors_debug.py` for line-by-line inspection.<br>4. Run the NGSolve notebook on the same MSH/JSON fixture and the MATLAB script on the same MAT fixture.<br>5. Run the full `compare` and `timing` commands from `README.md`. |
